# **AI Piano Transcriptor & Typesetting Pipeline**

This interactive notebook allows you to transcribe piano solo performances into sheet music and MIDI using deep learning and custom typesetting heuristics.

### **How to Use:**
1. Run the **Setup** cell to install LilyPond and Python dependencies.
2. Run the **Configuration & Input** cell to specify the audio source (either a URL like YouTube/SoundCloud/Direct Audio, or a local file upload) and typeset settings.
3. Run the **Run Pipeline** cell to transcribe and compile the sheet music.
4. View the PDF sheet music preview, play the interactive piano roll visualizer, and download the output files.

## **1. Setup Environment**
Install LilyPond, FluidSynth, and Python dependencies, then clone the repository. Enabling Google Drive integration allows you to cache pip dependencies and downloaded audio files, saving time on subsequent runs.

In [ ]:
#@title Install Dependencies (takes ~1-2 minutes) { display-mode: "form" }
import os
import sys
import shutil

Use_Google_Drive = False #@param {type:"boolean"}
pip_cache = ""

if Use_Google_Drive:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Create directories for caching
    drive_base = "/content/drive/MyDrive/PianoTranscriptor"
    pip_cache = os.path.join(drive_base, ".pip_cache")
    os.makedirs(pip_cache, exist_ok=True)
    os.makedirs(os.path.join(drive_base, "Cache"), exist_ok=True)
    os.makedirs(os.path.join(drive_base, "Outputs"), exist_ok=True)
    print(f"Google Drive directories initialized at: {drive_base}")

# Check if system packages are already installed
if not shutil.which('lilypond') or not shutil.which('fluidsynth'):
    print("Installing system packages (LilyPond, FluidSynth, Poppler, ffmpeg)...")
    !apt-get update -qq
    !apt-get install -y -qq lilypond fluidsynth poppler-utils ffmpeg &>/dev/null
else:
    print("System packages (LilyPond, FluidSynth, ffmpeg) are already installed.")

# Check if python dependencies are already installed
deps_installed = False
try:
    import pretty_midi
    import librosa
    import torch
    import piano_transcription_inference
    import soundfile
    import scipy
    import mido
    import pdf2image
    import yt_dlp
    deps_installed = True
    print("Python dependencies are already installed.")
except ImportError:
    pass

if not deps_installed:
    print("Installing Python packages...")
    cache_opt = f"--cache-dir {pip_cache}" if (Use_Google_Drive and pip_cache) else ""
    !pip install {cache_opt} -q pretty_midi librosa torch piano-transcription-inference soundfile scipy mido pdf2image
    !pip install {cache_opt} -q -U yt-dlp
    print("Python dependencies installed successfully!")

if not os.path.exists('ai_piano_transcriptor'):
    print("Cloning pipeline repository...")
    !git clone https://github.com/rfarnham/ai_piano_transcriptor.git

if '/content/ai_piano_transcriptor' not in sys.path:
    sys.path.append('/content/ai_piano_transcriptor')

print("Setup complete!")

## **2. Input Settings & Media Extraction**
Configure your transcription settings and select your input source. You can either paste a **URL** (YouTube, SoundCloud, direct audio link, etc.) or **upload a file** directly from your local computer.

In [ ]:
#@title Input Configuration { display-mode: "form" }
import urllib.parse
import subprocess
import os
import hashlib
from google.colab import files

# Ensure working directory is /content so downloads/uploads are placed correctly
if os.path.exists('/content'):
    os.chdir('/content')

Title = "My Transcription" #@param {type:"string"}
Composer = "AI Transcriptor" #@param {type:"string"}
BPM = 112 #@param {type:"integer"}
Key_Flats = 0 #@param {type:"slider", min:0, max:7, step:1}
Key_Sharps = 0 #@param {type:"slider", min:0, max:7, step:1}

Source = "URL" #@param ["URL", "Upload Local File"]
URL = "https://www.youtube.com/watch?v=L3WXPd6_IuA" #@param {type:"string"}

audio_filename = None

def get_url_cache_name(url_str):
    if 'v=' in url_str:
        video_id = url_str.split('v=')[1].split('&')[0]
        return f"yt_{video_id}.wav"
    elif 'youtu.be/' in url_str:
        video_id = url_str.split('youtu.be/')[1].split('?')[0]
        return f"yt_{video_id}.wav"
    else:
        h = hashlib.md5(url_str.encode('utf-8')).hexdigest()
        return f"audio_{h[:12]}.wav"

if Source == "URL":
    url = URL.strip()
    if not url:
        print("Error: Please enter a valid URL.")
    else:
        cache_name = get_url_cache_name(url)
        local_path = os.path.join('/content', cache_name)
        drive_path = None
        
        if 'Use_Google_Drive' in globals() and Use_Google_Drive:
            drive_path = os.path.join('/content/drive/MyDrive/PianoTranscriptor/Cache', cache_name)
            
        if drive_path and os.path.exists(drive_path):
            print(f"Found cached audio in Google Drive: {drive_path}")
            import shutil
            shutil.copy(drive_path, local_path)
            audio_filename = local_path
        elif os.path.exists(local_path):
            print(f"Found cached audio in local notebook directory: {local_path}")
            audio_filename = local_path
        else:
            parsed = urllib.parse.urlparse(url)
            path_lower = parsed.path.lower()
            is_direct_audio = any(path_lower.endswith(ext) for ext in ['.mp3', '.wav', '.m4a', '.ogg', '.flac'])
            
            print(f"Downloading audio content from: {url}...")
            download_success = False
            
            if is_direct_audio:
                import urllib.request
                try:
                    urllib.request.urlretrieve(url, local_path)
                    print(f"Direct audio file downloaded successfully to: {local_path}")
                    download_success = True
                except Exception as e:
                    print(f"Direct download failed ({e}), falling back to yt-dlp...")
                    is_direct_audio = False
                    
            if not is_direct_audio:
                print("Extracting audio using yt-dlp with client spoofing (Android)...")
                cmd = [
                    "yt-dlp", 
                    "-x", 
                    "--audio-format", "wav", 
                    "--audio-quality", "0", 
                    "--no-keep-video",
                    "--extractor-args", "youtube:player-client=android",
                    url, 
                    "-o", local_path
                ]
                res = subprocess.run(cmd, capture_output=True, text=True)
                if res.returncode == 0:
                    actual_path = local_path
                    if not os.path.exists(actual_path) and os.path.exists(actual_path + ".wav"):
                        actual_path += ".wav"
                    
                    if os.path.exists(actual_path):
                        if actual_path != local_path:
                            os.rename(actual_path, local_path)
                        print(f"Audio successfully extracted and saved: {local_path}")
                        download_success = True
                else:
                    print("Error extracting audio with yt-dlp:")
                    print(res.stderr)
                    print("\nTroubleshooting: Try re-running Setup, or use a different link.")
            
            if download_success:
                audio_filename = local_path
                if drive_path:
                    import shutil
                    print(f"Caching downloaded audio to Google Drive...")
                    shutil.copy(local_path, drive_path)
            else:
                audio_filename = None
else:
    print("Please upload your local audio file...")
    uploaded = files.upload()
    if not uploaded:
        print("Error: No file uploaded.")
    else:
        uploaded_name = list(uploaded.keys())[0]
        audio_filename = os.path.abspath(uploaded_name)
        print(f"File uploaded successfully: {audio_filename}")

## **3. Run Pipeline**
Runs transcription inference, warps rubato beat times to a steady grid, applies adaptive quantization, statefully splits treble/bass staves, and typeset the final sheet music score.

In [ ]:
import os
import subprocess

if not audio_filename:
    print("Error: Missing input audio file. Please run cell 2 first.")
else:
    # Resolve absolute path before switching directory
    input_audio_path = os.path.abspath(audio_filename)
    if not os.path.exists(input_audio_path):
        # Fallback to /content if not found
        input_audio_path = os.path.join('/content', os.path.basename(audio_filename))
        
    if not os.path.exists(input_audio_path):
        print(f"Error: Audio file not found at {input_audio_path}. Please run cell 2 again.")
    else:
        # Change directory to repository
        %cd /content/ai_piano_transcriptor
        
        cmd = [
            "python3", "main.py",
            "--input-audio", input_audio_path,
            "--output-ly", "transcription.ly",
            "--bpm", str(BPM),
            "--key-flats", str(Key_Flats),
            "--key-sharps", str(Key_Sharps),
            "--title", Title,
            "--composer", Composer
        ]
        
        print("Starting transcription pipeline (this may take a few minutes)...")
        res = subprocess.run(cmd, capture_output=True, text=True)
        print(res.stdout)
        
        if res.returncode != 0:
            print("Pipeline Error:")
            print(res.stderr)
        else:
            print("Compiling LilyPond source to PDF and MIDI...")
            compile_res = subprocess.run(["lilypond", "transcription.ly"], capture_output=True, text=True)
            print(compile_res.stdout)
            if compile_res.returncode == 0:
                print("Compilation successful!")
                
                # Copy outputs to Google Drive if enabled
                if 'Use_Google_Drive' in globals() and Use_Google_Drive:
                    import shutil
                    sanitized_title = "".join(c for c in Title if c.isalnum() or c in (' ', '_', '-')).rstrip()
                    folder_name = f"{sanitized_title}_{BPM}"
                    drive_output_dir = os.path.join("/content/drive/MyDrive/PianoTranscriptor/Outputs", folder_name)
                    os.makedirs(drive_output_dir, exist_ok=True)
                    print(f"Saving outputs to Google Drive: {drive_output_dir}")
                    for out_file in ["transcription.pdf", "transcription.midi", "warped_synth.wav"]:
                        src = os.path.join("/content/ai_piano_transcriptor", out_file)
                        if os.path.exists(src):
                            shutil.copy(src, drive_output_dir)
                            print(f"Saved: {out_file}")
            else:
                print("LilyPond compilation issues:")
                print(compile_res.stderr)
                
        # Change directory back to content
        %cd /content

## **4. View Sheet Music & Interactive Playback**
View the first page of the typeset sheet music and use the interactive piano roll visualizer to play and watch notes highlight as they are played.

In [ ]:
import os
import base64
from IPython.display import Image, display, HTML
from pdf2image import convert_from_path

pdf_path = "/content/ai_piano_transcriptor/transcription.pdf"
midi_path = "/content/ai_piano_transcriptor/transcription.midi"

# 1. Display score preview page 1
if os.path.exists(pdf_path):
    pages = convert_from_path(pdf_path, first_page=1, last_page=1)
    if pages:
        preview_img = "/content/first_page_preview.png"
        pages[0].save(preview_img, "PNG")
        print("\n--- Sheet Music Preview (Page 1) ---")
        display(Image(preview_img, width=800))
else:
    print("PDF file not found. Make sure the pipeline ran successfully.")

# 2. Render Interactive MIDI Player and Piano Roll Visualizer
if os.path.exists(midi_path):
    with open(midi_path, "rb") as f:
        encoded_midi = base64.b64encode(f.read()).decode('utf-8')
    midi_uri = f"data:audio/midi;base64,{encoded_midi}"
    
    html_code = f"""
    <!-- Load Tone.js, Magenta core, and html-midi-player scripts -->
    <script src="https://cdn.jsdelivr.net/combine/npm/tone@14.7.58,npm/@magenta/music@1.23.1/es6/core.js,npm/html-midi-player@1.5.0"></script>
    
    <div style="font-family: sans-serif; margin: 30px 0; max-width: 800px; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #fcfcfc;">
      <h3 style="margin-top: 0; color: #333;">--- Interactive Piano Roll Visualizer ---</h3>
      <p style="font-size: 14px; color: #666; margin-bottom: 15px;">Press Play below to listen to the transcribed performance and watch notes highlight on the piano keyboard roll.</p>
      
      <midi-player
        src="{midi_uri}"
        sound-font="https://storage.googleapis.com/magentadata/js/soundfonts/sgm_plus"
        visualizer="#my-visualizer"
        style="width: 100%; display: block; margin-bottom: 15px;">
      </midi-player>
      
      <midi-visualizer
        id="my-visualizer"
        type="piano-roll"
        src="{midi_uri}"
        style="width: 100%; height: 300px; border: 1px solid #ccc; border-radius: 4px; display: block; background: #fafafa;">
      </midi-visualizer>
    </div>
    """
    display(HTML(html_code))
else:
    print("MIDI file not found. Visualizer could not be loaded.")

## **5. Download Outputs**
Download the final typeset PDF score, compiled MIDI file, and synthesized WAV audio.

In [ ]:
from google.colab import files
import os

print("Preparing download links...")
base_dir = "/content/ai_piano_transcriptor"
download_files = ["transcription.pdf", "transcription.midi", "warped_synth.wav"]

for f in download_files:
    full_path = os.path.join(base_dir, f)
    if os.path.exists(full_path):
        print(f"Downloading: {f}")
        files.download(full_path)